# Review Pipeline V1 Single-Case Run

Use this notebook to run one docket through the new review pipeline agent scaffold and inspect the full input and output trace in MLflow.

What it does:
- loads one docket's full catalog-backed Doc Intelligence exports without character caps
- assembles a prompt-ready review bundle
- runs the agent once with a two-stage instruction: calculate from sentencing information first, then substantiate with facts from anywhere in the docket
- asks for each offense-level step in claim, sentencing-evidence, guideline-support, and justification form
- logs inputs, prompt preview, artifact output, and message trace to MLflow

In [1]:
from __future__ import annotations

from datetime import UTC, datetime
import json
from pathlib import Path
import sys

import mlflow
import mlflow.langchain
import pandas as pd

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from review_pipeline_v1.agent import (
    build_review_agent,
    build_review_prompt,
    extract_review_artifact,
    merge_case_facts_into_artifact,
    run_case_facts_extraction,
    run_docket_review,
    serialize_agent_messages,
)
from review_pipeline_v1.runtime import (
    resolve_execution_env,
    resolve_mlflow_experiment,
    resolve_model_name,
    resolve_spark_session,
)
from review_pipeline_v1.search_tools import build_search_tools
from review_pipeline_v1.single_case import (
    build_review_input_from_docintel,
    read_optional_text_file,
    review_input_to_dict,
)

c:\Users\sonutka\main\2-Envs\tam-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EXECUTION_ENV = None
DOCINTEL_ROOT = "/Volumes/usdo_aa_catalog/research_tam_datasets/federal_sentencing/cases/docintel_text"
# Example available dockets from the current catalog view: 16648584, 16661467, 16673142, 16681714, 16685478, 16688439, 16690637
DOCKET_ID = "16661467"
GUIDELINE_YEAR = 2024
MODEL_NAME_OVERRIDE = None
MLFLOW_EXPERIMENT_OVERRIDE = None
MLFLOW_RUN_NAME = None
PROMPT_SUFFIX_FILE = None
MAX_DOCUMENTS = None
MAX_CHARS_PER_DOCUMENT = None
MAX_CASE_SUMMARY_CHARS = None
MAX_AGENT_STEPS = 60
SHOW_PROMPT_CHARS = None
EXTRA_REVIEWER_CONTEXT = {
    "calculation_workflow": "stage_1_sentencing_info_calculation_stage_2_global_fact_substantiation",
    "step_output_style": "For each offense_level_step, separate the claim, sentencing evidence, guideline support, and justification. Prefer short guideline text or tax-table row excerpts when available.",
}

In [3]:
execution_env = EXECUTION_ENV or resolve_execution_env()
model_name = MODEL_NAME_OVERRIDE or resolve_model_name()
mlflow_experiment = MLFLOW_EXPERIMENT_OVERRIDE or resolve_mlflow_experiment()
spark = resolve_spark_session(app_name="review-pipeline-v1-single-case-notebook")
prompt_suffix = read_optional_text_file(PROMPT_SUFFIX_FILE)

review_input = build_review_input_from_docintel(
    docket_id=DOCKET_ID,
    execution_env=execution_env,
    docintel_root=DOCINTEL_ROOT,
    spark=spark,
    guideline_year=GUIDELINE_YEAR,
    max_documents=MAX_DOCUMENTS,
    max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
    max_case_summary_chars=MAX_CASE_SUMMARY_CHARS,
    reviewer_context=EXTRA_REVIEWER_CONTEXT,
)
review_input_payload = review_input_to_dict(review_input)
prompt_text = build_review_prompt(review_input, prompt_suffix=prompt_suffix)

print(json.dumps({
    "execution_env": execution_env,
    "model_name": model_name,
    "mlflow_experiment": mlflow_experiment,
    "docket_id": DOCKET_ID,
    "selected_document_count": len(review_input_payload.get("selected_documents", [])),
    "case_summary_chars": len(review_input_payload.get("case_summary") or ""),
}, indent=2))

pd.DataFrame(review_input_payload.get("selected_documents", []))

{
  "execution_env": "local",
  "model_name": "openai:gpt-5",
  "mlflow_experiment": "/Users/sonutka@mfcgd.com/review-pipeline-v1-single-case",
  "docket_id": "16661467",
  "selected_document_count": 4,
  "case_summary_chars": 201711
}


,document_id,source_file_name,source_pdf_path,docintel_path,document_role,why_selected,page_count,content_length,document_text
0,123348641_fact_Statement_of_Offense,123348641_fact_Statement_of_Offense.pdf,/Volumes/usdo_aa_catalog/research_tam_datasets...,\Volumes\usdo_aa_catalog\research_tam_datasets...,unknown,Loaded from the docket's catalog Doc Intellige...,7,9687,Case 1:20-cr-00006-KBJ Document 11 Filed 02/07...
1,123348650_fact_Plea_Agreement,123348650_fact_Plea_Agreement.pdf,/Volumes/usdo_aa_catalog/research_tam_datasets...,\Volumes\usdo_aa_catalog\research_tam_datasets...,unknown,Loaded from the docket's catalog Doc Intellige...,13,34918,Case 1:20-cr-00006-KBJ Document 10 Filed 02/07...
2,146702486_sentencing_memo_Sentencing_Memorandum,146702486_sentencing_memo_Sentencing_Memorandu...,/Volumes/usdo_aa_catalog/research_tam_datasets...,\Volumes\usdo_aa_catalog\research_tam_datasets...,unknown,Loaded from the docket's catalog Doc Intellige...,21,39208,Case 1:20-cr-00006-KBJ Document 21 Filed 09/24...
3,146737682_sentencing_memo_Sentencing_Memorandum,146737682_sentencing_memo_Sentencing_Memorandu...,/Volumes/usdo_aa_catalog/research_tam_datasets...,\Volumes\usdo_aa_catalog\research_tam_datasets...,unknown,Loaded from the docket's catalog Doc Intellige...,57,117052,Case 1:20-cr-00006-KBJ Document 23 Filed 09/24...


In [4]:
if SHOW_PROMPT_CHARS is None:
    print(prompt_text)
else:
    print(prompt_text[:SHOW_PROMPT_CHARS])

You are reviewing one federal sentencing docket.

Return only valid JSON matching this schema exactly:
{
  "docket_id": "string",
  "queue_label": "ready_for_review | needs_manual_followup | insufficient_evidence",
  "selected_documents": [
    {
      "document_id": "string",
      "source_file_name": "string",
      "document_role": "government_sentencing_memo | defense_sentencing_memo | plea_agreement | stipulation | presentence_report | other | unknown",
      "why_selected": "string"
    }
  ],
  "case_facts": [
    {
      "fact": "string",
      "source_document": "string",
      "evidence_excerpt": "string",
      "support_strength": "strong | medium | weak"
    }
  ],
  "offense_level_steps": [
    {
      "step_name": "string",
      "guideline_reference": "string | null",
      "proposed_adjustment": "integer | null",
      "explanation": "string",
      "source_document": "string",
      "evidence_excerpt": "string",
      "support_strength": "strong | medium | weak",
     

In [4]:
def to_jsonable(value):
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    if isinstance(value, tuple):
        return [to_jsonable(item) for item in value]
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass
    return value

if mlflow.active_run() is not None:
    mlflow.end_run()
mlflow.set_experiment(mlflow_experiment)
mlflow.langchain.autolog(log_traces=True, silent=True)
run_name = MLFLOW_RUN_NAME or f"review_pipeline_v1_single_case_{datetime.now(UTC).strftime('%Y%m%d_%H%M%S')}"

agent_tools = build_search_tools(guideline_year=GUIDELINE_YEAR)
agent = build_review_agent(model=model_name, tools=agent_tools)
with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_params({
        "pipeline": "review_pipeline_v1",
        "execution_env": execution_env,
        "model_name": model_name,
        "docket_id": str(DOCKET_ID),
        "guideline_year": GUIDELINE_YEAR,
        "docintel_root": DOCINTEL_ROOT,
        "max_documents": "all" if MAX_DOCUMENTS is None else MAX_DOCUMENTS,
        "max_chars_per_document": "unlimited" if MAX_CHARS_PER_DOCUMENT is None else MAX_CHARS_PER_DOCUMENT,
        "max_case_summary_chars": "unlimited" if MAX_CASE_SUMMARY_CHARS is None else MAX_CASE_SUMMARY_CHARS,
        "max_agent_steps": MAX_AGENT_STEPS,
        "prompt_suffix_file": PROMPT_SUFFIX_FILE or "",
        "tool_count": len(agent_tools),
        "case_facts_enrichment": "enabled",
    })
    mlflow.set_tags({
        "pipeline": "review_pipeline_v1_single_case",
        "tracking": "mlflow",
        "input_source": "catalog_docintel",
    })

    with mlflow.start_span(name="single_case_review_pipeline_v1") as case_span:
        case_span.set_inputs({
            "docket_id": str(DOCKET_ID),
            "guideline_year": GUIDELINE_YEAR,
            "model_name": model_name,
            "selected_documents": review_input_payload.get("selected_documents", []),
            "case_summary": review_input_payload.get("case_summary") or "",
            "prompt_text": prompt_text,
        })
        primary_result = run_docket_review(
            agent,
            review_input,
            prompt_suffix=prompt_suffix,
            config={"recursion_limit": MAX_AGENT_STEPS},
        )
        primary_artifact = extract_review_artifact(primary_result)
        case_facts_result = run_case_facts_extraction(
            agent,
            review_input,
            primary_artifact,
            config={"recursion_limit": MAX_AGENT_STEPS},
        )
        case_facts_payload = extract_review_artifact(case_facts_result)
        artifact = merge_case_facts_into_artifact(primary_artifact, case_facts_payload)
        serialized_messages = serialize_agent_messages(primary_result.get("messages", []))
        case_facts_messages = serialize_agent_messages(case_facts_result.get("messages", []))
        case_span.set_outputs({
            "artifact": artifact,
            "message_count": len(serialized_messages),
            "case_facts_message_count": len(case_facts_messages),
        })

    summary = {
        "run_id": run.info.run_id,
        "docket_id": str(DOCKET_ID),
        "review_input": review_input_payload,
        "primary_artifact": primary_artifact,
        "case_facts_payload": case_facts_payload,
        "artifact": artifact,
        "message_count": len(serialized_messages),
        "case_facts_message_count": len(case_facts_messages),
    }
    mlflow.log_dict(to_jsonable(summary), "single_case/review_pipeline_v1_summary.json")
    mlflow.log_dict(to_jsonable({"prompt_text": prompt_text}), "single_case/review_pipeline_v1_prompt.json")
    mlflow.log_dict(to_jsonable({"messages": serialized_messages}), "single_case/review_pipeline_v1_messages.json")
    mlflow.log_dict(to_jsonable({"messages": case_facts_messages}), "single_case/review_pipeline_v1_case_facts_messages.json")

print(json.dumps({
    "run_id": run.info.run_id,
    "docket_id": str(DOCKET_ID),
    "tool_count": len(agent_tools),
    "primary_case_facts_count": len(primary_artifact.get("case_facts") or []),
    "enriched_case_facts_count": len(artifact.get("case_facts") or []),
    "final_total_offense_level": artifact.get("final_total_offense_level"),
}, indent=2))

🏃 View run review_pipeline_v1_single_case_20260520_074037 at: https://adb-2498687920111167.7.azuredatabricks.net/ml/experiments/2910463209424510/runs/56d4fecd6c144a12adb0644c04200e74
🧪 View experiment at: https://adb-2498687920111167.7.azuredatabricks.net/ml/experiments/2910463209424510
{
  "run_id": "56d4fecd6c144a12adb0644c04200e74",
  "docket_id": "16661467",
  "tool_count": 1,
  "primary_case_facts_count": 17,
  "enriched_case_facts_count": 86,
  "final_total_offense_level": 23
}


In [8]:
artifact

{'docket_id': '16661467',
 'docket_support_status': 'supported',
 'queue_label': 'ready_for_review',
 'selected_documents': [{'document_id': '123348641_fact_Statement_of_Offense',
   'source_file_name': '123348641_fact_Statement_of_Offense.pdf',
   'document_role': 'stipulation',
   'why_selected': 'Signed factual basis for the plea; provides detailed tax-loss figures and timeframes.'},
  {'document_id': '123348650_fact_Plea_Agreement',
   'source_file_name': '123348650_fact_Plea_Agreement.pdf',
   'document_role': 'plea_agreement',
   'why_selected': 'Contains the parties’ guideline calculation, total tax loss, relevant-conduct stipulation, and acceptance terms.'},
  {'document_id': '146702486_sentencing_memo_Sentencing_Memorandum',
   'source_file_name': '146702486_sentencing_memo_Sentencing_Memorandum.pdf',
   'document_role': 'government_sentencing_memo',
   'why_selected': 'Reiterates the guideline math (base level 26, total offense level 23) and ties to PSR; summarizes loss figur

In [7]:
artifact_shape_check = {
    "has_open_questions": "open_questions" in artifact,
    "has_reviewer_notes": "reviewer_notes" in artifact,
    "docket_support_status": artifact.get("docket_support_status"),
    "queue_label": artifact.get("queue_label"),
    "final_total_offense_level": artifact.get("final_total_offense_level"),
    "offense_level_steps": [
        {
            "step_name": step.get("step_name"),
            "claim": step.get("claim"),
            "included_in_total_offense_level": step.get("included_in_total_offense_level"),
            "sentencing_evidence_excerpt": (((step.get("sentencing_evidence") or [{}])[0]).get("evidence_excerpt") or "")[:180],
            "guideline_support_reference": ((step.get("guideline_support") or [{}])[0]).get("guideline_reference"),
            "guideline_support_excerpt": (((step.get("guideline_support") or [{}])[0]).get("guideline_text_excerpt") or "")[:180],
        }
        for step in (artifact.get("offense_level_steps") or [])
    ],
}

print(json.dumps(artifact_shape_check, indent=2))

{
  "has_open_questions": false,
  "has_reviewer_notes": false,
  "docket_support_status": "supported",
  "queue_label": "ready_for_review",
  "final_total_offense_level": 23,
  "offense_level_steps": [
    {
      "step_name": "base_offense_level",
      "claim": "Base offense level 26 because total tax loss, including relevant conduct, is $10,747,370.13 (exceeds $9,500,000).",
      "included_in_total_offense_level": true,
      "sentencing_evidence_excerpt": "\"The parties agree that the total tax loss, including relevant conduct is $10,747,370.13, corresponding to a base offense level of 26 under U.S.S.G. \u00a7 2T1.6 and 2T4.1...\"",
      "guideline_support_reference": "\u00a72T1.6",
      "guideline_support_excerpt": "Base Offense Level: Level from \u00a72T4.1 (Tax Table) corresponding to the tax not collected or accounted for and paid over."
    },
    {
      "step_name": "relevant_conduct_sales_tax",
      "claim": "Include $6,285,955.17 in DC sales-tax losses as relevant con